# Day-ahead consumption forecast: actual vs forecast temperature

Every day at 12:00 we submit an hourly consumption forecast for the following day (D+1).
Temperature is the main driver of demand, so the question here is how much accuracy we lose
by using a temperature *forecast* instead of the actual temperature.

Three variants are compared:

- actual temperature at the target hour,
- the latest available weather forecast for the target hour,
- the noon-issued forecast (what we would have when we submit).

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
fc = pd.read_csv("../data/weather_forecasts.csv", parse_dates=["forecast_datetime"])

print(df.shape, fc.shape)
df.head()

(17520, 6) (70004, 4)


,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71
3,2022-01-01 03:00:00+00:00,25381.3,-0.75,7.05,0.0,70.92
4,2022-01-01 04:00:00+00:00,25223.0,-0.09,7.36,0.0,60.78


In [2]:
fc.head()

,origin_datetime,forecast_datetime,horizon_h,temp_forecast_c
0,2022-01-01 00:00:00+00:00,2022-01-01 01:00:00+00:00,1,0.51
1,2022-01-01 00:00:00+00:00,2022-01-01 02:00:00+00:00,2,-1.43
2,2022-01-01 00:00:00+00:00,2022-01-01 03:00:00+00:00,3,0.40
3,2022-01-01 00:00:00+00:00,2022-01-01 04:00:00+00:00,4,0.29
4,2022-01-01 00:00:00+00:00,2022-01-01 05:00:00+00:00,5,-1.05


In [3]:
fc.describe()

,horizon_h,temp_forecast_c
count,70004.000000,70004.000000
mean,24.489401,10.244223
std,13.853456,6.982999
min,1.000000,-11.480000
25%,12.000000,4.750000
50%,24.000000,10.270000
75%,36.000000,15.690000
max,48.000000,34.510000


## Calendar features and target

The target is consumption 24 hours ahead. Calendar features use local time because the
demand profile follows the clock, not UTC. Consumption at the same hour on the previous day
is the usual persistence feature.

In [4]:
df["time_local"] = df["time"].dt.tz_convert("Europe/London")
df["hour"] = df["time_local"].dt.hour
df["dow"] = df["time_local"].dt.dayofweek
df["weekend"] = (df["dow"] >= 5).astype(int)

df["target"] = df["consumption_mwh"].shift(-24)
df["temp_target"] = df["temp_c"].shift(-24)          # temperature at the target hour
df["cons_lag"] = df["consumption_mwh"]               # same hour, previous day (relative to target)
df["cons_lag_week"] = df["consumption_mwh"].shift(144)  # same hour, previous week (relative to target)

df[["time", "time_local", "hour", "target", "temp_target", "cons_lag", "cons_lag_week"]].head()

,time,time_local,hour,target,temp_target,cons_lag,cons_lag_week
0,2022-01-01 00:00:00+00:00,2022-01-01 00:00:00+00:00,0,26785.3,0.15,26858.4,NaN
1,2022-01-01 01:00:00+00:00,2022-01-01 01:00:00+00:00,1,25826.5,-0.59,26177.8,NaN
2,2022-01-01 02:00:00+00:00,2022-01-01 02:00:00+00:00,2,25799.8,-1.09,26229.4,NaN
3,2022-01-01 03:00:00+00:00,2022-01-01 03:00:00+00:00,3,26081.4,-0.86,25381.3,NaN
4,2022-01-01 04:00:00+00:00,2022-01-01 04:00:00+00:00,4,25860.6,-0.54,25223.0,NaN


## Forecast temperature

Forecasts are issued twice a day (00:00 and 12:00) for the next 48 hours. For each target hour
we take the most recent forecast available (`temp_fc_latest`), and separately the noon-issued
forecast (`temp_fc_noon`).

In [5]:
fc["forecast_datetime"] = fc["forecast_datetime"].dt.tz_localize(None)

latest = (
    fc.sort_values("origin_datetime")
      .groupby("forecast_datetime")
      .last()[["temp_forecast_c"]]
      .rename(columns={"temp_forecast_c": "temp_fc_latest"})
)

noon = (
    fc[fc["origin_datetime"].str.contains("12:00")]
      [["forecast_datetime", "temp_forecast_c"]]
      .rename(columns={"temp_forecast_c": "temp_fc_noon"})
)

print(latest.shape, noon.shape)
latest.head(3)

(17519, 1) (34990, 2)


,temp_fc_latest
forecast_datetime,
2022-01-01 01:00:00,0.51
2022-01-01 02:00:00,-1.43
2022-01-01 03:00:00,0.40


In [6]:
df["target_time"] = df["time_local"].dt.tz_localize(None) + pd.Timedelta(hours=24)

df = df.merge(latest, left_on="target_time", right_index=True, how="left")
df = df.merge(noon, left_on="target_time", right_on="forecast_datetime", how="left")

print(df.shape)
df[["time", "target_time", "temp_target", "temp_fc_latest", "temp_fc_noon"]].head()

(35003, 18)


,time,target_time,temp_target,temp_fc_latest,temp_fc_noon
0,2022-01-01 00:00:00+00:00,2022-01-02 00:00:00,0.15,0.31,0.31
1,2022-01-01 01:00:00+00:00,2022-01-02 01:00:00,-0.59,-0.12,-1.74
2,2022-01-01 02:00:00+00:00,2022-01-02 02:00:00,-1.09,-1.59,-1.35
3,2022-01-01 03:00:00+00:00,2022-01-02 03:00:00,-0.86,-0.77,-1.01
4,2022-01-01 04:00:00+00:00,2022-01-02 04:00:00,-0.54,-0.14,-0.75


In [7]:
df["temp_fc_noon"] = df["temp_fc_noon"].bfill()
df = df.dropna()
print(len(df), "rows after cleaning;", df.isna().sum().sum(), "NaNs left")

34704 rows after cleaning; 0 NaNs left


## Train / test split

80/20 random split, fixed seed for reproducibility.

In [8]:
rng = np.random.default_rng(0)
is_train = rng.random(len(df)) < 0.8

hour_dummies = pd.get_dummies(df["hour"], prefix="h").astype(float)


def design(temp_col):
    X = pd.concat([hour_dummies, df[["weekend", "cons_lag", "cons_lag_week", temp_col]]], axis=1)
    return X


print(is_train.sum(), "train rows,", (~is_train).sum(), "test rows")

27788 train rows, 6916 test rows


## Models

One linear model per temperature variant. Same calendar and persistence features throughout.

In [9]:
results = {}
models = {}
for name, col in [("actual temp", "temp_target"),
                  ("latest forecast", "temp_fc_latest"),
                  ("noon forecast", "temp_fc_noon")]:
    X = design(col)
    y = df["target"]
    model = LinearRegression().fit(X[is_train], y[is_train])
    pred = model.predict(X[is_train])
    results[name] = {
        "R2": r2_score(y[is_train], pred),
        "MAE": mean_absolute_error(y[is_train], pred),
        "temp_coef": model.coef_[-1],
    }
    models[name] = model

res = pd.DataFrame(results).T
res

,R2,MAE,temp_coef
actual temp,0.919185,952.709016,-138.138359
latest forecast,0.918733,955.791757,-134.903176
noon forecast,0.915733,972.395163,-108.280282


## Results

In [10]:
loss_latest = (res.loc["actual temp", "R2"] - res.loc["latest forecast", "R2"]) / res.loc["actual temp", "R2"]
loss_noon = (res.loc["actual temp", "R2"] - res.loc["noon forecast", "R2"]) / res.loc["actual temp", "R2"]

print(res.round(4))
print()
print(f"R2 lost by using the latest forecast instead of actual temperature: {100 * loss_latest:.2f}%")
print(f"R2 lost by using the noon forecast:                               {100 * loss_noon:.2f}%")

                     R2       MAE  temp_coef
actual temp      0.9192  952.7090  -138.1384
latest forecast  0.9187  955.7918  -134.9032
noon forecast    0.9157  972.3952  -108.2803

R2 lost by using the latest forecast instead of actual temperature: 0.05%
R2 lost by using the noon forecast:                               0.38%


Using the latest forecast costs almost nothing versus actual temperature, and the noon forecast is
only slightly worse. Recommendation: use the latest available forecast in production; the
temperature forecast is not the bottleneck of this model.